Zadanie 5 – LEFT JOIN
Używając tabel z zadania 4:
Wymagania:
Wykonaj LEFT JOIN pokazujący WSZYSTKICH klientów (nawet bez zamówień)
Użyj COALESCE aby zamienić NULL na 0 w kwocie zamówienia
Znajdź klientów którzy NIE złożyli zamówienia (WHERE order_id IS NULL)

In [19]:
import pandas as pd
import numpy as np
import time
import sqlite3
conn = sqlite3.connect(":memory:")

df_customers =  pd.DataFrame({
    'customer_id': [1, 2, 3, 4, 5],
    'name': ['Jan', 'Paweł', 'Anna', 'Marcin', 'Jakub'],
    'city': ['Czersk', 'Warszawa', 'Sopot', 'Hel', 'Białystok']
})

df_orders = pd.DataFrame({
    'order_id': [101, 102, 103, 104, 105, 106, 107, 108],
    'customer_id': [1, 2, 3, 2, 1, 1, 3, 2],
    'amount': [280.0, 130.5, 430.0, 190.5, 70.0, 220.0, 350.5, 90.0]
})

df_customers.to_sql("customers", conn, index=False, if_exists="replace")
df_orders.to_sql("orders", conn, index=False, if_exists="replace")

query_all_customers = """
SELECT
    c.customer_id,
    c.name,
    c.city,
    o.order_id,
    COALESCE(o.amount, 0) AS order_amount
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
ORDER BY c.customer_id, o.order_id
"""

all_customers = pd.read_sql_query(query_all_customers, conn)
print("Wszyscy klienci:")
print(all_customers)

query_without_orders = """
SELECT
    c.customer_id,
    c.name,
    c.city
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL
ORDER BY c.customer_id
"""

customers_without_orders = pd.read_sql_query(query_without_orders, conn)
print("\nKlienci bez zamówień:")
print(customers_without_orders)


Wszyscy klienci:
   customer_id    name       city  order_id  order_amount
0            1     Jan     Czersk     101.0         280.0
1            1     Jan     Czersk     105.0          70.0
2            1     Jan     Czersk     106.0         220.0
3            2   Paweł   Warszawa     102.0         130.5
4            2   Paweł   Warszawa     104.0         190.5
5            2   Paweł   Warszawa     108.0          90.0
6            3    Anna      Sopot     103.0         430.0
7            3    Anna      Sopot     107.0         350.5
8            4  Marcin        Hel       NaN           0.0
9            5   Jakub  Białystok       NaN           0.0

Klienci bez zamówień:
   customer_id    name       city
0            4  Marcin        Hel
1            5   Jakub  Białystok


Zadanie 6 – Podstawowy GROUP BY
Używając tabeli
orders
Wymagania:
Policz liczbę zamówień dla każdego klienta
Oblicz łączną kwotę zamówień dla każdego klienta
Oblicz średnią wartość zamówienia dla każdego klienta

In [6]:
query_order_summary = """
SELECT
    customer_id,
    COUNT(order_id) AS number_of_orders,
    SUM(amount) AS total_amount,
    ROUND(AVG(amount), 2) AS average_amount
FROM orders
GROUP BY customer_id
ORDER BY customer_id
"""

order_summary = pd.read_sql_query(query_order_summary, conn)
order_summary

,customer_id,number_of_orders,total_amount,average_amount
0,1,3,570.0,190.00
1,2,3,411.0,137.00
2,3,2,780.5,390.25


Zadanie 7 – COUNT i SUM
Wymagania:
Policz łączną liczbę produktów w tabeli
products
Oblicz łączną wartość zapasów (stock
price) dla wszystkich produktów
Oblicz średnią cenę produktów w każdej kategorii

In [15]:
df_products = pd.DataFrame({
'product_id': [1, 2, 3, 4, 5],
'name': ['Laptop', 'Mysz', 'Klawiatura', 'Monitor', 'Słuchawki'],
'category': ['Elektronika', 'Akcesoria', 'Akcesoria', 'Elektronika', 'Akcesoria'],
'price': [3500, 50, 200, 1200, 150],
'stock': [10, 150, 80, 25, 60]
})

df_products.to_sql("products", conn, index=False, if_exists="replace")

query_products_total = """
SELECT
    COUNT(*) AS total_products,
    SUM(stock * price) AS total_inventory_value
FROM products
"""

products_total = pd.read_sql_query(query_products_total, conn)
print("Łączna liczba produktów:")
print(products_total)


query_average_by_category = """
SELECT
    category,
    ROUND(AVG(price), 2) AS average_price
FROM products
GROUP BY category
ORDER BY category
"""

average_by_category = pd.read_sql_query(query_average_by_category, conn)
print("\nŚrednia cena kategorii:")
print(average_by_category)



Łączna liczba produktów:
   total_products  total_inventory_value
0               5                  97500

Średnia cena kategorii:
      category  average_price
0    Akcesoria         133.33
1  Elektronika        2350.00


Zadanie 8 – Tworzenie indeksu,
EXPLAIN QUERY PLAN
i pomiar czasu
Wymagania:
Utwórz tabelę z 10 000 rekordami (użyj NumPy z L10)
Zmierz czas zapytania
SELECT ... WHERE kolumna = ...
BEZ indeksu
za pomocą
time.perf_counter()
(uśrednij z 5 powtórzeń)
Uruchom
EXPLAIN QUERY PLAN
dla tego zapytania – zapisz plan (zobaczysz
SCAN
)
Utwórz indeks na kolumnie filtrowanej (
CREATE INDEX ...
)
Zmierz czas zapytania
Z indeksem
(też uśrednij z 5 powtórzeń)
Uruchom
EXPLAIN QUERY PLAN
ponownie – powinieneś zobaczyć
SEARCH ... USING INDEX ...
Porównaj: czas BEZ / Z, przyspieszenie (x razy), różnicę w planach

In [20]:
conn_bench = sqlite3.connect(':memory:')
table = pd.DataFrame({
    'id': range(1, 10001),
    'value': np.random.randint(1, 10000, 10000)
})

table.to_sql('bench_table', conn_bench, index=False)

# Bez indeksu

start = time.perf_counter()

pd.read_sql(
    "SELECT * FROM bench_table WHERE value = 5000",
    conn_bench
)

end = time.perf_counter()

print(f"Bez indeksu: {end - start:.8f} s")

# Z indeksem
cursor = conn_bench.cursor()

cursor.execute("""
CREATE INDEX idx_value
ON bench_table(value)
""")

conn_bench.commit()


start = time.perf_counter()

pd.read_sql(
    "SELECT * FROM bench_table WHERE value = 5000",
    conn_bench
)

end = time.perf_counter()

print(f"Z indeksem: {end - start:.8f} s")

Bez indeksu: 0.00241030 s
Z indeksem: 0.00041950 s


Zadanie 18 – Self-JOIN
Wymagania:
Utwórz tabelę
employees
z kolumnami: employee_id, name, manager_id
Użyj self-JOIN aby wyświetlić pracownika i jego managera:
SELECT e1.name AS employee, e2.name AS manager FROM employees e1 LEFT JOIN employees e2 ONe1.manager_id = e2.employee_id
Znajdź wszystkich pracowników bez managera (TOP management)
Policz liczbę podwładnych dla każdego managera

In [22]:
employees = pd.DataFrame({
    "employee_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "name": [
        "Anna Dyrektor",
        "Piotr Kierownik",
        "Marek Kierownik",
        "Ewa Programistka",
        "Jan Analityk",
        "Sara Programistka",
        "Adam Księgowy",
        "Lena Testerka"
    ],
    "manager_id": [None, 1, 1, 2, 2, 3, 3, 3]
})

employees.to_sql("employees", conn, index=False, if_exists="replace")

# Ta sama tabela występuje dwa razy: e1 to pracownik, e2 to manager.
query_employee_manager = """
SELECT
    e1.name AS employee,
    e2.name AS manager
FROM employees e1
LEFT JOIN employees e2 ON e1.manager_id = e2.employee_id
ORDER BY e1.employee_id
"""

employee_manager = pd.read_sql_query(query_employee_manager, conn)
print("Pracownicy i ich managerowie:")
print(employee_manager)

# Brak manager_id oznacza najwyższy poziom zarządzania.
query_top_management = """
SELECT
    employee_id,
    name
FROM employees
WHERE manager_id IS NULL
"""

top_management = pd.read_sql_query(query_top_management, conn)
print("\nPracownicy bez managera:")
print(top_management)

# m oznacza managera, a e jego podwładnego.
query_subordinates = """
SELECT
    m.name AS manager,
    COUNT(e.employee_id) AS number_of_subordinates
FROM employees m
INNER JOIN employees e ON e.manager_id = m.employee_id
GROUP BY m.employee_id, m.name
ORDER BY number_of_subordinates DESC, m.name
"""

subordinates = pd.read_sql_query(query_subordinates, conn)
print("\nLiczba podwładnych u managera:")
print(subordinates)

Pracownicy i ich managerowie:
            employee          manager
0      Anna Dyrektor              NaN
1    Piotr Kierownik    Anna Dyrektor
2    Marek Kierownik    Anna Dyrektor
3   Ewa Programistka  Piotr Kierownik
4       Jan Analityk  Piotr Kierownik
5  Sara Programistka  Marek Kierownik
6      Adam Księgowy  Marek Kierownik
7      Lena Testerka  Marek Kierownik

Pracownicy bez managera:
   employee_id           name
0            1  Anna Dyrektor

Liczba podwładnych u managera:
           manager  number_of_subordinates
0  Marek Kierownik                       3
1    Anna Dyrektor                       2
2  Piotr Kierownik                       2
